# 07 — Understand and compare the runs

This notebook explains the validation results one step at a time.

> **Question: At the same replay budget, does MFR remember earlier behaviors better than random replay without learning the new behavior less well?**

The locked test data stays closed until every planned run and the analysis plan are finished.


## 1. What do Order 1 and Order 2 mean?

One model learns three behaviors in sequence.

| Order | Stage 1 | Stage 2 | Stage 3 |
|---|---|---|---|
| **Order 1** | Helpful | Safe | Quality |
| **Order 2** | Safe | Helpful | Quality |

Using two orders checks whether the result changes when Helpful and Safe switch places. Quality is last in both orders.

| Method | Simple meaning |
|---|---|
| **No replay** | Use only the current behavior. |
| **Random 10%** | Use 18 new pairs and 2 random old pairs. |
| **Random 14.3%** | Use 18 new pairs and 3 random old pairs. |
| **Lowest margin** | Replay old pairs the model currently finds difficult. |
| **MFR 10%** | Replay old pairs whose score fell most since they were learned. |


## 2. Setup

Change `FOCUS_ORDER` and `FOCUS_SEED` to inspect one matched group. Tables are normal HTML, not the large interactive viewer, so they are easier to select and copy.

Set `SAVE_OUTPUTS = True` only when you want CSV tables and PNG figures saved in Drive.


In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

FOCUS_ORDER = 1
FOCUS_SEED = 0
SAVE_OUTPUTS = False

if Path('/content').exists():
    if not Path('/content/mfr-dpo').exists():
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    !git -C /content/mfr-dpo pull -q
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = Path('/content/mfr-dpo')
    DRIVE_DIR = Path('/content/drive/MyDrive/CSCI544/mfr-dpo')
else:
    REPO = next((p for p in [Path.cwd(), Path.cwd().parent]
                 if (p/'configs/experiment_protocol.json').exists()), Path.cwd())
    DRIVE_DIR = Path(os.environ.get('MFR_DRIVE_DIR', REPO.parent/'drive'))

protocol = json.loads((REPO/'configs/experiment_protocol.json').read_text())
TOLERANCE = protocol['new_learning_tolerance_points']
OUTPUT_DIR = DRIVE_DIR/'analysis/notebook_07'
METHODS = ['none','random','random_high','lowest_margin','mfr']
MLABEL = {'none':'No replay','random':'Random 10%','random_high':'Random 14.3%',
          'lowest_margin':'Lowest margin','mfr':'MFR 10%'}
MCOLOR = {'none':'#666666','random':'#377bd1','random_high':'#7557c7',
          'lowest_margin':'#20a878','mfr':'#ef6a32'}
DATASETS = ['helpful','safe','quality']
DLABEL = {'helpful':'Helpful','safe':'Safe','quality':'Quality'}
DCOLOR = {'helpful':'#377bd1','safe':'#d14b61','quality':'#20a878'}

def show_table(df, name):
    out=df.copy()
    nums=out.select_dtypes(include='number').columns
    out[nums]=out[nums].round(2)
    display(HTML(out.to_html(index=False,border=0,na_rep='—',classes='result-table')))
    if SAVE_OUTPUTS:
        OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
        out.to_csv(OUTPUT_DIR/f'{name}.csv',index=False)

def finish(fig,name):
    fig.tight_layout()
    if SAVE_OUTPUTS:
        OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
        fig.savefig(OUTPUT_DIR/f'{name}.png',dpi=200,bbox_inches='tight')
    plt.show()

plt.rcParams.update({'figure.dpi':105,'font.size':9,'axes.spines.top':False,'axes.spines.right':False})
display(HTML('''<style>
.result-table{border-collapse:collapse;font-size:13px;margin:8px 0 16px}
.result-table th{background:#eee;font-weight:600}
.result-table th,.result-table td{border:1px solid #bbb;padding:5px 9px;text-align:center}
</style>'''))
print('Reading:', DRIVE_DIR/'runs')


## 3. Load the completed v2 runs

Only run folders containing `COMPLETE.json`, `settings.json`, and `results.csv` are used. Pilot runs are ignored. This first small table shows what is available.


In [ ]:
frames=[]; records={}
for folder in sorted((DRIVE_DIR/'runs').glob('*')):
    needed=[folder/'COMPLETE.json',folder/'settings.json',folder/'results.csv']
    if not all(p.exists() for p in needed): continue
    s=json.loads((folder/'settings.json').read_text())
    if str(s.get('protocol_version'))!='2.0': continue
    r={'run_name':s.get('run_name',folder.name),'order_id':int(s['order_id']),
       'order':list(s['order']),'method':str(s['method']),'seed':int(s['seed']),
       'folder':folder,'stage1_from':s.get('stage1_from')}
    records[r['run_name']]=r
    f=pd.read_csv(folder/'results.csv')
    for key in ['run_name','order_id','method','seed']: f[key]=r[key]
    frames.append(f)
if not frames: raise FileNotFoundError('No completed v2 runs were found')
runs=pd.concat(frames,ignore_index=True)

inventory=pd.DataFrame([{'Order':r['order_id'],'Sequence':' → '.join(DLABEL[x] for x in r['order']),
                         'Seed':r['seed'],'Method':MLABEL[r['method']]} for r in records.values()])
show_table(inventory.sort_values(['Order','Seed','Method']),'completed_runs')

focus_records={n:r for n,r in records.items() if r['order_id']==FOCUS_ORDER and r['seed']==FOCUS_SEED}
focus=runs[runs.run_name.isin(focus_records)].copy()
if focus.empty: raise ValueError(f'No completed runs for order {FOCUS_ORDER}, seed {FOCUS_SEED}')
focus_methods=[m for m in METHODS if m in set(focus.method)]
order=protocol['orders'][str(FOCUS_ORDER)]
print('Selected:', ' → '.join(DLABEL[x] for x in order), '| seed',FOCUS_SEED)
print('Methods:', ', '.join(MLABEL[m] for m in focus_methods))


## 4. What does accuracy mean?

Each validation example has a prompt, a preferred answer, and a rejected answer. A pair counts as correct when training moved the model toward the preferred answer relative to the frozen base model.

- Higher accuracy is better.
- Each validation dataset has 200 pairs, so 0.5 percentage point is one pair.
- This is preference accuracy, not proof that generated answers are truly safe, helpful, or correct.


## 5. Step 1 — Did each stage learn its behavior?

The gain is `score after the stage − score before the stage`. All three datasets are included because every dataset is trained once. Larger positive gain means stronger learning during that stage.


In [ ]:
learn=[]
for name,r in focus_records.items():
    part=focus[focus.run_name==name]
    for stage,d in enumerate(r['order'],1):
        before=part[(part.stage==stage-1)&(part.eval_set==d)]
        after=part[(part.stage==stage)&(part.eval_set==d)]
        if len(before) and len(after):
            learn.append({'method':r['method'],'dataset':d,
                          'gain':float(after.iloc[0].accuracy-before.iloc[0].accuracy)})
learning=pd.DataFrame(learn)
wide=learning.pivot(index='method',columns='dataset',values='gain').reindex(focus_methods).reindex(columns=DATASETS)
t=wide.reset_index().rename(columns={'method':'Method',**{d:f'{DLABEL[d]} gain' for d in DATASETS}})
t.Method=t.Method.map(MLABEL); show_table(t,'learning_gain')

x=np.arange(len(focus_methods)); w=.22
fig,ax=plt.subplots(figsize=(8,3.5))
for i,d in enumerate(DATASETS): ax.bar(x+(i-1)*w,wide[d],w,label=DLABEL[d],color=DCOLOR[d])
ax.axhline(0,color='black',lw=.8); ax.grid(axis='y',alpha=.2)
ax.set_xticks(x,[MLABEL[m] for m in focus_methods],rotation=15,ha='right')
ax.set_ylabel('Accuracy gained'); ax.set_title(f'Learning each behavior — order {FOCUS_ORDER}, seed {FOCUS_SEED}')
ax.legend(ncol=3,frameon=False); finish(fig,'01_learning_gain')


## 6. Step 2 — Follow all three behaviors through training

Each small chart includes all three datasets. To keep the figure readable, it shows only the three main methods: No replay, equal-budget Random 10%, and MFR 10%. The secondary methods remain in all result tables and the next chart.

A line should rise when its behavior is trained. A later fall is forgetting.


In [ ]:
core=[m for m in ['none','random','mfr'] if m in focus_methods]
fig,axes=plt.subplots(1,len(core),figsize=(3.2*len(core),3.3),sharey=True)
axes=np.atleast_1d(axes)
for ax,m in zip(axes,core):
    for d in DATASETS:
        p=focus[(focus.method==m)&(focus.eval_set==d)].sort_values('stage')
        ax.plot(p.stage,p.accuracy,marker='o',lw=1.8,color=DCOLOR[d],label=DLABEL[d])
    ax.set_xticks([0,1,2,3],['Base']+[f'After\n{DLABEL[d]}' for d in order],fontsize=8)
    ax.set_title(MLABEL[m]); ax.grid(axis='y',alpha=.2)
axes[0].set_ylabel('Preference accuracy (%)'); axes[-1].legend(frameon=False,fontsize=8)
fig.suptitle(f'All three behaviors — order {FOCUS_ORDER}, seed {FOCUS_SEED}',y=1.03)
finish(fig,'02_behavior_journey')


## 7. Step 3 — What did each final model know?

This is the easiest result to show first. It compares all methods after all three stages. Every graph here includes Helpful, Safe, and Quality. The average is useful as a quick summary, but the three separate scores still matter.


In [ ]:
final=(focus[focus.stage==3].pivot_table(index='method',columns='eval_set',values='accuracy',aggfunc='first')
       .reindex(focus_methods).reindex(columns=DATASETS))
final['average']=final.mean(axis=1)
t=final.reset_index().rename(columns={'method':'Method','helpful':'Helpful','safe':'Safe','quality':'Quality','average':'Average'})
t.Method=t.Method.map(MLABEL); show_table(t,'final_scores')

x=np.arange(len(focus_methods)); w=.22
fig,ax=plt.subplots(figsize=(8,3.5))
for i,d in enumerate(DATASETS): ax.bar(x+(i-1)*w,final[d],w,label=DLABEL[d],color=DCOLOR[d])
ax.set_xticks(x,[MLABEL[m] for m in focus_methods],rotation=15,ha='right')
ax.set_ylabel('Final preference accuracy (%)'); ax.set_title(f'Final scores — order {FOCUS_ORDER}, seed {FOCUS_SEED}')
ax.legend(ncol=3,frameon=False); ax.grid(axis='y',alpha=.2); finish(fig,'03_final_scores')


## 8. Step 4 — How much was forgotten?

Forgetting is `final score − score immediately after that behavior was learned`. Zero means no forgetting. A negative number means the score fell; closer to zero is better.

**Why one dataset is missing from this graph:** it shows only the first two behaviors in the selected order. Quality is last in both orders, so no later training happens after it. Its retention cannot be measured. The table still includes all three columns and marks the final dataset as “Not measurable.”


In [ ]:
ret=[]
for name,r in focus_records.items():
    p=focus[focus.run_name==name]
    for stage,d in enumerate(r['order'][:-1],1):
        a=p[(p.stage==stage)&(p.eval_set==d)]; z=p[(p.stage==3)&(p.eval_set==d)]
        if len(a) and len(z): ret.append({'run_name':name,'method':r['method'],'dataset':d,
                                         'change':float(z.iloc[0].accuracy-a.iloc[0].accuracy)})
retention=pd.DataFrame(ret)
rwide=retention.pivot(index='method',columns='dataset',values='change').reindex(focus_methods)
t=pd.DataFrame({'Method':[MLABEL[m] for m in focus_methods]})
for d in DATASETS: t[DLABEL[d]]=rwide[d].to_numpy() if d in rwide else 'Not measurable'
show_table(t,'forgetting')

old=order[:-1]; x=np.arange(len(focus_methods)); w=.32
fig,ax=plt.subplots(figsize=(8,3.5))
for i,d in enumerate(old): ax.bar(x+(i-.5)*w,rwide[d],w,label=DLABEL[d],color=DCOLOR[d])
ax.axhline(0,color='black',lw=.8); ax.grid(axis='y',alpha=.2)
ax.set_xticks(x,[MLABEL[m] for m in focus_methods],rotation=15,ha='right')
ax.set_ylabel('Accuracy change after learning'); ax.set_title(f'Forgetting of earlier behaviors — order {FOCUS_ORDER}, seed {FOCUS_SEED}')
ax.legend(frameon=False); finish(fig,'04_forgetting')


## 9. Step 5 — Did MFR beat equal-budget random?

Less forgetting alone is not enough; a method might simply block new learning. The first table shows retention, the final task, and the final average together.

The frozen rule compares MFR 10% with Random 10%. MFR passes when it retains old behavior better and its final-task score is no more than 2 points below Random 10%.


In [ ]:
summary=[]
for name,r in records.items():
    p=runs[runs.run_name==name]; changes=[]
    for stage,d in enumerate(r['order'][:-1],1):
        a=p[(p.stage==stage)&(p.eval_set==d)]; z=p[(p.stage==3)&(p.eval_set==d)]
        if len(a) and len(z): changes.append(float(z.iloc[0].accuracy-a.iloc[0].accuracy))
    z=p[p.stage==3]; task=z[z.eval_set==r['order'][-1]]
    if len(changes)==2 and len(task):
        summary.append({'run_name':name,'order_id':r['order_id'],'seed':r['seed'],'method':r['method'],
                        'old_change':np.mean(changes),'final_task':float(task.iloc[0].accuracy),
                        'final_average':z.accuracy.mean()})
summary=pd.DataFrame(summary)
s=summary[(summary.order_id==FOCUS_ORDER)&(summary.seed==FOCUS_SEED)].set_index('method').reindex(focus_methods)
t=s.reset_index()[['method','old_change','final_task','final_average']].rename(columns={
  'method':'Method','old_change':'Mean old-behavior change','final_task':f'Final {DLABEL[order[-1]]} score','final_average':'Final average of all 3'})
t.Method=t.Method.map(MLABEL); show_table(t,'headline_balance')

rules=[]
for (o,seed),g in summary.groupby(['order_id','seed']):
    g=g.set_index('method')
    if not {'random','mfr'}<=set(g.index): continue
    ra=g.loc['mfr','old_change']-g.loc['random','old_change']
    ft=g.loc['mfr','final_task']-g.loc['random','final_task']
    rules.append({'Order':o,'Seed':seed,'MFR retention advantage':ra,'MFR final-task difference':ft,
                  'Retains better?':'Yes' if ra>0 else 'No','Within 2-point limit?':'Yes' if ft>=-TOLERANCE else 'No',
                  'Passes rule?':'Yes' if ra>0 and ft>=-TOLERANCE else 'No'})
show_table(pd.DataFrame(rules),'mfr_success_rule')

# Short sentences you can use when explaining this result.
print('\nSimple reading of this selected group:')
if 'none' in s.index:
    print(f"• Without replay, the average old-behavior change was {s.loc['none','old_change']:.2f} points.")
if {'random','mfr'} <= set(s.index):
    advantage = s.loc['mfr','old_change'] - s.loc['random','old_change']
    task_gap = s.loc['mfr','final_task'] - s.loc['random','final_task']
    print(f"• MFR retained {advantage:.2f} points better than equal-budget random.")
    print(f"• MFR finished {task_gap:.2f} points {'higher' if task_gap >= 0 else 'lower'} on the final task.")
best_retention = s['old_change'].idxmax()
best_average = s['final_average'].idxmax()
print(f"• {MLABEL[best_retention]} had the best pure retention in this group.")
print(f"• {MLABEL[best_average]} had the best final average across all three datasets.")


## 10. Step 6 — Is the difference larger than the uncertainty?

The bootstrap resamples the same 200 pairs and gives a 95% interval for MFR minus each baseline. Positive favors MFR. If the interval crosses zero, that comparison is not conclusive.

**Why Quality is absent:** this is a retention comparison. Quality has no later stage, so it has no retention value. Its final score was shown in Steps 3 and 5.


In [ ]:
def stage_folder(r,stage):
    d=r['order'][stage-1]; direct=r['folder']/f'stage{stage}_{d}'
    if direct.exists(): return direct
    if stage==1 and r['stage1_from']:
        source=Path(r['stage1_from'])
        if not source.exists(): source=r['folder'].parent/source.name
        return source/f'stage1_{d}'
    return direct

def pair_change(r,stage,d):
    a=pd.read_csv(stage_folder(r,stage)/f'margins_{d}_val.csv')[['id','margin']].rename(columns={'margin':'a'})
    z=pd.read_csv(stage_folder(r,3)/f'margins_{d}_val.csv')[['id','margin']].rename(columns={'margin':'z'})
    q=a.merge(z,on='id'); q['change']=(q.z>0).astype(int)-(q.a>0).astype(int)
    return q[['id','change']]

boots=[]
for baseline in ['random','random_high','lowest_margin']:
    by_method={r['method']:r for r in focus_records.values()}
    if not {'mfr',baseline}<=set(by_method): continue
    for stage,d in enumerate(order[:-1],1):
        a=pair_change(by_method['mfr'],stage,d); b=pair_change(by_method[baseline],stage,d)
        q=a.merge(b,on='id',suffixes=('_mfr','_base'))
        diff=100*(q.change_mfr-q.change_base).to_numpy()
        rng=np.random.default_rng(100+stage+10*METHODS.index(baseline))
        samples=rng.choice(diff,size=(5000,len(diff)),replace=True).mean(axis=1)
        lo,hi=np.quantile(samples,[.025,.975])
        boots.append({'Baseline':MLABEL[baseline],'Behavior':DLABEL[d],'MFR difference':diff.mean(),
                      '95% low':lo,'95% high':hi,'Conclusive?':'Yes' if lo>0 or hi<0 else 'No'})
boot=pd.DataFrame(boots); show_table(boot,'bootstrap_intervals')
if len(boot):
    y=np.arange(len(boot)); center=boot['MFR difference'].to_numpy()
    fig,ax=plt.subplots(figsize=(8,.38*len(boot)+1.6))
    ax.errorbar(center,y,xerr=[center-boot['95% low'],boot['95% high']-center],fmt='o',color=MCOLOR['mfr'],capsize=3)
    ax.axvline(0,color='black',lw=1); ax.set_yticks(y,[f'{r.Baseline} — {r.Behavior}' for _,r in boot.iterrows()]); ax.invert_yaxis()
    ax.set_xlabel('MFR minus baseline retention (points)'); ax.set_title('Uncertainty for earlier behaviors'); ax.grid(axis='x',alpha=.2)
    finish(fig,'05_uncertainty')


## 11. Step 7 — What was replayed?

Replay slots count repeats; unique pairs count different examples. Uses per pair shows how concentrated replay was.

The source table includes all three datasets. Quality is zero because it is the current task in Stage 3 and there is no Stage 4 where it could become old replay data.


In [ ]:
rows=[]
for r in focus_records.values():
    for stage in [2,3]:
        path=stage_folder(r,stage)/'replay_log.csv'
        if not path.exists(): continue
        log=pd.read_csv(path)
        if log.empty: continue
        n=log.id.nunique(); counts=log.dataset.value_counts()
        rows.append({'Method':MLABEL[r['method']],'Stage':stage,'Learning now':DLABEL[r['order'][stage-1]],
                     'Replay slots':len(log),'Unique pairs':n,'Uses per pair':len(log)/n,
                     'From Helpful':int(counts.get('helpful',0)),'From Safe':int(counts.get('safe',0)),'From Quality':int(counts.get('quality',0))})
replay=pd.DataFrame(rows)
show_table(replay[['Method','Stage','Learning now','Replay slots','Unique pairs','Uses per pair']],'replay_amount')
show_table(replay[['Method','Stage','From Helpful','From Safe','From Quality']],'replay_sources')


## 12. Step 8 — What did it cost?

The first table and chart include time from all three training datasets. The second table shows total examples, replay examples, and extra candidate-scoring time.


In [ ]:
rows=[]
for r in focus_records.values():
    mins={d:0. for d in DATASETS}; total=scoring=0.; steps=examples=replayed=0
    for stage,d in enumerate(r['order'],1):
        path=stage_folder(r,stage)/'history.csv'
        if not path.exists(): continue
        h=pd.read_csv(path); mins[d]=float(h.minutes.max()); total+=mins[d]; steps+=len(h)
        if 'scoring_minutes' in h: scoring+=float(h.scoring_minutes.max())
        if 'n_new' in h:
            nr=int(h.n_replay.sum()) if 'n_replay' in h else 0
            examples+=int(h.n_new.sum())+nr; replayed+=nr
    rows.append({'key':r['method'],'Method':MLABEL[r['method']],**{f'{DLABEL[d]} minutes':mins[d] for d in DATASETS},
                 'Total minutes':total,'Scoring minutes':scoring,'Steps':steps,'Examples':examples,'Replay examples':replayed})
cost=pd.DataFrame(rows); cost['key']=pd.Categorical(cost.key,METHODS,ordered=True); cost=cost.sort_values('key')
show_table(cost[['Method','Helpful minutes','Safe minutes','Quality minutes','Total minutes']],'runtime')
show_table(cost[['Method','Scoring minutes','Steps','Examples','Replay examples']],'cost_budget')
fig,ax=plt.subplots(figsize=(8,3.5)); bottom=np.zeros(len(cost))
for d in DATASETS:
    v=cost[f'{DLABEL[d]} minutes'].to_numpy(); ax.bar(cost.Method,v,bottom=bottom,label=DLABEL[d],color=DCOLOR[d]); bottom+=v
ax.set_ylabel('Minutes'); ax.set_title(f'Runtime by stage — order {FOCUS_ORDER}, seed {FOCUS_SEED}'); ax.tick_params(axis='x',rotation=15)
ax.legend(ncol=3,frameon=False); ax.grid(axis='y',alpha=.2); finish(fig,'06_runtime')


## 13. Step 9 — Summary across every completed cell

This table updates as runs finish. The full study has four cells: Order 1/Seed 0, Order 1/Seed 1, Order 2/Seed 0, and Order 2/Seed 1. Do not make the final claim from one cell.


In [ ]:
overall=(summary.groupby('method').agg(**{'Cells completed':('run_name','count'),
  'Average old-behavior change':('old_change','mean'),'Average final-task score':('final_task','mean'),
  'Average final score across all 3':('final_average','mean')}).reindex(METHODS).dropna(how='all').reset_index().rename(columns={'method':'Method'}))
overall.Method=overall.Method.map(MLABEL); show_table(overall,'all_completed_cells')


## 14. How to explain the results

Use this order when presenting:

1. Explain Order 1 and Order 2.
2. Show that each stage learned its behavior.
3. Show final Helpful, Safe, and Quality scores.
4. Show forgetting, and explain why the last behavior has no retention measurement.
5. Compare MFR 10% with Random 10% using the frozen rule.
6. Show the uncertainty. A small difference is not enough when its interval crosses zero.
7. Explain what MFR replayed and how much extra time scoring required.
8. State that a final conclusion needs both seeds and both orders, followed by locked-test, generation, and human evaluation.

A strong final result means MFR repeatedly retains earlier behaviors better than equal-budget random replay while staying within the 2-point new-learning limit. Every individual cell and confidence interval should still be reported.
